# Trabajo Práctico 2 (TP2) - Parte 1: Preprocesamiento de Datos y Análisis Estadístico Exploratorio

### Machine Learning 1 (23433)
#### Facultad de Ingeniería - Universidad Nacional de Asunción (FIUNA)

---

## Objetivos de la Parte 1
1. Cargar e inspeccionar la estructura del conjunto de datos unificado de rendimiento académico de FIUNA (`reglamento_nuevo_unificado.csv`).
2. Mapear las 27 siglas e intensificaciones curriculares a las 7 carreras principales de la facultad.
3. Aplicar un preprocesamiento de integridad de datos **no destructivo** que preserve el 100% de los 64,295 registros sin eliminar filas.
4. Responder a 6 preguntas estadísticas exploratorias clave para auditar la masa estudiantil, la retención académica y las tasas de aprobación.


In [1]:
# 1. Carga de Librerías Fundamentales
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 1000)

# Carga del Dataset
csv_path = 'reglamento_nuevo_unificado.csv'
if not os.path.exists(csv_path):
    csv_path = os.path.join('..', 'Clase_5', 'reglamento_nuevo_unificado.csv')

df_raw = pd.read_csv(csv_path)
print(f"Dataset cargado exitosamente: {df_raw.shape[0]:,} filas y {df_raw.shape[1]} columnas.")


Dataset cargado exitosamente: 64,295 filas y 29 columnas.


## Ejercicio 1: Mapeo de Intensificaciones Curriculares a Carreras Principales

Completa el diccionario `career_code_mapping` para mapear las 27 siglas del sistema (`CIV-PLS13`, `INT9CONSTR`, `ELE-PLS23`, `INT9SDIGYT`, `MCT-PLS13`, `IND-PLS13`, `CGF-PLS13`, `MEC-PLS13`, `ECA-PLS13`, etc.) a sus 7 carreras principales:
- `Ing. Civil`
- `Ing. Electrónica`
- `Ing. Mecatrónica`
- `Ing. Industrial`
- `Ing. Geográfica`
- `Ing. Mecánica`
- `Ing. Electromecánica`


In [2]:
# TODO: Completar la estructura del diccionario career_code_mapping
career_code_mapping = {
    # === ESCRIBE TU CÓDIGO AQUÍ ===
    'CIV-PLS13': 'Ing. Civil', 'CIV-PLS23': 'Ing. Civil', 'INT9CONSTR': 'Ing. Civil',
    'INT9TRANSP': 'Ing. Civil', 'INT9ORTERR': 'Ing. Civil', 'INT9SANEHI': 'Ing. Civil',
    
    'ELE-PLS13': 'Ing. Electrónica', 'ELE-PLS23': 'Ing. Electrónica',
    'INT9ELECTR': 'Ing. Electrónica', 'INT9SDIGYT': 'Ing. Electrónica',
    
    'MCT-PLS13': 'Ing. Mecatrónica', 'MCT-PLS23': 'Ing. Mecatrónica', 'MCT9-OPT': 'Ing. Mecatrónica',
    
    'IND-PLS13': 'Ing. Industrial', 'IND-PLS23': 'Ing. Industrial',
    'INT9G-ECO': 'Ing. Industrial', 'INT9-PROYT': 'Ing. Industrial',
    
    'CGF-PLS13': 'Ing. Geográfica', 'CGF-PLS23': 'Ing. Geográfica', 'INT9RNYMA': 'Ing. Geográfica',
    
    'MEC-PLS13': 'Ing. Mecánica', 'MEC-PLS23': 'Ing. Mecánica',
    'INT9MECANI': 'Ing. Mecánica', 'MEC9-OPT': 'Ing. Mecánica',
    
    'ECA-PLS13': 'Ing. Electromecánica', 'ECA-PLS23': 'Ing. Electromecánica', 'ECA9-OPT': 'Ing. Electromecánica'
}

df_clean = df_raw.copy()
# TODO: Asignar la nueva columna Carrera_Nombre usando map()
df_clean['Carrera_Nombre'] = df_clean['Cod.Car.Sec'].astype(str).str.strip().map(career_code_mapping)

# Definición del Target Binario (1 para 'S', 0 para 'N')
df_clean = df_clean.dropna(subset=['Aprobado', 'Carrera_Nombre']).copy()
df_clean['Target'] = (df_clean['Aprobado'] == 'S').astype(int)

print(f"Filas tras mapeo de carreras: {len(df_clean):,}")


Filas tras mapeo de carreras: 64,295


## Ejercicio 2: Limpieza Numérica y Generación de Atributos Derivados (Sin Eliminación de Filas)

Convierte los campos numéricos de parciales y evaluaciones a formato float utilizando `pd.to_numeric(..., errors='coerce')` para mantener el 100% de las filas.
Crea las siguientes variables derivadas:
- `Score_Parciales`: Promedio entre el 1er y 2do parcial.
- `Diff_Parciales`: Diferencia ($2^\circ\text{Par} - 1^\circ\text{Par}$).


In [8]:
# TODO: Limpiar y convertir atributos numéricos sin eliminar filas
# === ESCRIBE TU CÓDIGO AQUÍ ===
df_clean['Primer_Par_Clean'] = pd.to_numeric(df_clean['Primer.Par'], errors='coerce')
df_clean['Segundo_Par_Clean'] = pd.to_numeric(df_clean['Segundo.Par'], errors='coerce')
df_clean['Score_Parciales'] = (df_clean['Primer_Par_Clean'].fillna(0) + df_clean['Segundo_Par_Clean'].fillna(0)) / 2.0
df_clean['Diff_Parciales'] = df_clean['Segundo_Par_Clean'].fillna(0) - df_clean['Primer_Par_Clean'].fillna(0)
df_clean['TPLab_Clean'] = pd.to_numeric(df_clean['TPLab.'], errors='coerce')
df_clean['Asis_Clean'] = pd.to_numeric(df_clean['Asis'], errors='coerce')
df_clean['Firma_Clean'] = pd.to_numeric(df_clean['Firma'], errors='coerce')
df_clean['FirmaCalc_Clean'] = pd.to_numeric(df_clean['FirmaCalculada'], errors='coerce')

print(f"Filas preservadas en df_clean: {len(df_clean):,} (100% retención)")


Filas preservadas en df_clean: 64,295 (100% retención)


## Ejercicio 3: Auditoría Estadísticas Exploratoria (6 Preguntas Clave)

Responde a las siguientes 6 preguntas estadísticas utilizando código en Python sobre `df_raw` y `df_clean`:

1. **¿Cuántos estudiantes presentan registros en los tres ciclos del CSV?**
2. **¿Cuántas personas/registros tienen calificación final (`Nota.Final`)?**
3. **¿Cuántos tienen proceso (`FirmaCalculada` / `Firma`)?**
4. **¿Cuál es la tasa de aprobación global y por Carrera?**
5. **¿Cómo se comparan las notas medias del 1er y 2do Parcial según condición final (Aprobado vs No Aprobado)?**
6. **¿Cuál es la tasa de abandono / inasistencia total a parciales?**


In [10]:
# === 3. AUDITORÍA ESTADÍSTICA EXPLORATORIA DE FIUNA ===
print("=== 3. AUDITORÍA ESTADÍSTICA EXPLORATORIA DE FIUNA ===\n")

# 1. Estudiantes con registros en los 3 ciclos
df_raw['Ciclo'] = df_raw['Anho'].astype(str) + '-' + df_raw['Semestre'].astype(str)
student_ciclos = df_raw.groupby('ALUMNO_ID')['Ciclo'].nunique()
total_alumnos = df_raw['ALUMNO_ID'].nunique()
alumnos_3_ciclos = (student_ciclos == 3).sum()
pct_3_ciclos = (alumnos_3_ciclos / total_alumnos) * 100
print(f"1. Estudiantes con registros en los 3 ciclos: {alumnos_3_ciclos:,} ({pct_3_ciclos:.2f}% del total)")

# 2. Registros y personas con Nota.Final
nota_final_df = df_raw.dropna(subset=['Nota.Final'])
print(f"2. Registros con Nota.Final: {len(nota_final_df):,} | Estudiantes únicos: {nota_final_df['ALUMNO_ID'].nunique():,}")

# 3. Registros con FirmaCalculada / Firma >= 50
firma_calc_df = df_raw.dropna(subset=['FirmaCalculada'])
firma_50_df = df_clean[df_clean['Firma_Clean'] >= 50]
print(f"3. Registros con FirmaCalculada: {len(firma_calc_df):,} | Registros con Firma >= 50: {len(firma_50_df):,}")

# 4. Tasa de aprobación global y por Carrera
tasa_carrera = df_clean.groupby('Carrera_Nombre')['Target'].agg(
    Inscritos='count',
    Aprobados='sum'
)
tasa_carrera['Tasa_Aprob_Pct'] = (tasa_carrera['Aprobados'] / tasa_carrera['Inscritos']) * 100
print(f"\n4. Tasa de Aprobación por Carrera:\n{tasa_carrera}")

# 5. Promedio de parciales según condición final
parciales_target = df_clean.groupby('Target')[['Primer_Par_Clean', 'Segundo_Par_Clean']].mean()
print(f"\n5. Promedios de Parciales por Target:\n{parciales_target}")

# 6. Tasa de abandono / inasistencia total a parciales (0 en ambos parciales)
p1_zero = df_clean['Primer_Par_Clean'].fillna(0) == 0
p2_zero = df_clean['Segundo_Par_Clean'].fillna(0) == 0
both_zero = (p1_zero & p2_zero).sum()
pct_both_zero = (both_zero / len(df_clean)) * 100
print(f"\n6. Registros con 0 en ambos parciales: {both_zero:,} ({pct_both_zero:.2f}%)")


=== 3. AUDITORÍA ESTADÍSTICA EXPLORATORIA DE FIUNA ===

1. Estudiantes con registros en los 3 ciclos: 3,532 (74.22% del total)
2. Registros con Nota.Final: 42,720 | Estudiantes únicos: 4,344
3. Registros con FirmaCalculada: 19,407 | Registros con Firma >= 50: 41,715

4. Tasa de Aprobación por Carrera:
                      Inscritos  Aprobados  Tasa_Aprob_Pct
Carrera_Nombre                                            
Ing. Civil                29392      17202       58.526130
Ing. Electromecánica       1693       1150       67.926757
Ing. Electrónica          11928       6638       55.650570
Ing. Geográfica            4626       3275       70.795504
Ing. Industrial            6416       4050       63.123441
Ing. Mecatrónica           7147       4691       65.635931
Ing. Mecánica              3093       1999       64.629809

5. Promedios de Parciales por Target:
        Primer_Par_Clean  Segundo_Par_Clean
Target                                     
0              26.163306          19.56